In [1]:
!pip install seqeval

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for seqeval: filename=seqeval-1.2.2-py3-none-any.whl size=16162 sha256=994dd4747a8efc26cf69d95de95c057877ef274f926de149d9e6f0ebd5e5d69d
  Stored in directory: /root/.cache/pip/wheels/5f/b8/73/0b2c1a76b701a677653dd79ece07cfabd7457989dbfbdcd8d7
Successfully built seqeval


In [2]:
import os
import random
import warnings

from abc import ABC, abstractmethod
from dataclasses import dataclass

import numpy as np
import pandas as pd

import torch
import torch.nn.functional as F

from datasets import load_dataset
from huggingface_hub import login

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report
)

from seqeval.metrics import (
    classification_report as seqeval_report,
    f1_score as seqeval_f1
)

from tqdm.auto import tqdm

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM
)

warnings.filterwarnings("ignore")

In [3]:
@dataclass
class Config:

    # HF
    hf_token: str = ""

    # MODEL
    # model_name: str = "ura-hcmut/ura-llama-2.1-8b"
    model_name : str = "Qwen/Qwen3-8B"

    # INFERENCE
    batch_size: int = 8

    # GENERATION
    max_new_tokens: int = 32

    sample_ratio: float = 1
    # SEED
    seed: int = 42


cfg = Config()

In [4]:
def sample_dataframe(df, ratio=0.05, seed=42):

    return (
        df.sample(
            frac=ratio,
            random_state=seed
        )
        .reset_index(drop=True)
    )

In [5]:
class SeedManager:

    @staticmethod
    def set_seed(seed=42):

        random.seed(seed)
        np.random.seed(seed)

        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False


SeedManager.set_seed(cfg.seed)

In [6]:
login(cfg.hf_token)

In [7]:
class BaseDatasetProcessor(ABC):

    task_type = None

    def __init__(self, dataset_name):

        self.dataset_name = dataset_name
        self.dataset = None

    def load(self):

        self.dataset = load_dataset(
            self.dataset_name
        )

        return self

    @abstractmethod
    def get_test_dataframe(self):
        pass

In [8]:
class ViHSDProcessor(BaseDatasetProcessor):

    task_type = "classification"

    def __init__(self):

        super().__init__(
            "uitnlp/vihsd"
        )

        self.label_map = {
            0: "CLEAN",
            1: "OFFENSIVE",
            2: "HATE"
        }

    def get_test_dataframe(self):

        df = self.dataset["test"].to_pandas()

        df = df.rename(columns={
            "free_text": "text",
            "label_id": "label"
        })

        return df[["text", "label"]]

    def build_prompt(self, text: str):

        messages = [
            {
                "role": "system",
                "content": (
                    "Bạn là hệ thống phát hiện ngôn ngữ độc hại tiếng Việt.\n\n"
                    "Nhãn:\n"
                    "0 = CLEAN\n"
                    "1 = OFFENSIVE\n"
                    "2 = HATE\n\n"

                    "Định nghĩa:\n"
                    "- CLEAN: nội dung bình thường, không độc hại.\n"
                    "- OFFENSIVE: chửi tục, xúc phạm, công kích cá nhân, toxic nhưng không mang tính thù ghét nhóm/người.\n"
                    "- HATE: phát ngôn thù ghét, miệt thị, phân biệt vùng miền/tôn giáo/chủng tộc,"
                    " hoặc phản động/chống nhà nước/chính quyền.\n\n"
                    "- Kiểm tra theo thứ tự tính HATE, OFFENSIVE, CLEAN"

                    "Chỉ trả lời đúng duy nhất:\n"
                    "0 hoặc 1 hoặc 2"
                )
            },

            # FEW SHOT
            {
                "role": "user",
                "content": "bạn thật tuyệt vời"
            },
            {
                "role": "assistant",
                "content": "0"
            },

            {
                "role": "user",
                "content": "mày là đồ ngu"
            },
            {
                "role": "assistant",
                "content": "1"
            },

            {
                "role": "user",
                "content": "đồ súc vật mất dạy"
            },
            {
                "role": "assistant",
                "content": "1"
            },

            {
                "role": "user",
                "content": "hồ tặc hại dân hại nước"
            },
            {
                "role": "assistant",
                "content": "2"
            },

            {
                "role": "user",
                "content": "bọn mọi rợ nên cút khỏi đây"
            },
            {
                "role": "assistant",
                "content": "2"
            },

            {
                "role": "user",
                "content": "tôi yêu gia đình tôi"
            },
            {
                "role": "assistant",
                "content": "0"
            },

            # TARGET
            {
                "role": "user",
                "content": text
            }
        ]

        return messages

In [9]:
class ViCTSDProcessor(BaseDatasetProcessor):

    task_type = "classification"

    def __init__(self):

        super().__init__(
            "visolex/ViCTSD"
        )

        self.label_map = {
            0: "NON_TOXIC",
            1: "TOXIC"
        }

    def get_test_dataframe(self):

        df = (
            self.dataset["train"]
            .filter(lambda x: x["type"] == "test")
            .to_pandas()
        )

        df = df.rename(columns={
            "Comment": "text",
            "Toxicity": "label"
        })

        df = df[
            df["label"].isin([0, 1])
        ]

        return df[["text", "label"]]

    def build_prompt(self, text):
    
        return [
            {
                "role": "system",
                "content":
                    (
                        "Phân loại toxic comment tiếng Việt.\n\n"
    
                        "Nhãn:\n"
                        "0 = NON_TOXIC\n"
                        "1 = TOXIC\n\n"
    
                        "TOXIC bao gồm:\n"
                        "- chửi tục\n"
                        "- xúc phạm\n"
                        "- miệt thị\n"
                        "- công kích cá nhân\n"
                        "- sarcasm mang tính công kích\n"
                        "- toxic slang\n"
                        "- khinh thường\n"
                        "- toxic implicit theo ngữ cảnh\n"
                        "- lời nói mang tính hạ nhục hoặc ác ý\n\n"
    
                        "Một số toxic nhẹ hoặc cà khịa công kích "
                        "vẫn có thể là 1.\n\n"
    
                        "Không phải mọi câu tiêu cực đều là toxic.\n\n"
    
                        "Chỉ trả lời duy nhất:\n"
                        "0 hoặc 1"
                    )
            },
    
            # NON_TOXIC
            {
                "role": "user",
                "content": "Tôi thấy vụ này vẫn cần thêm thông tin"
            },
            {
                "role": "assistant",
                "content": "0"
            },
    
            {
                "role": "user",
                "content": "Không thích thì góp ý thôi chứ"
            },
            {
                "role": "assistant",
                "content": "0"
            },
    
            {
                "role": "user",
                "content": "Chắc do góc quay thôi"
            },
            {
                "role": "assistant",
                "content": "0"
            },
    
            {
                "role": "user",
                "content": "Hơi thất vọng thật"
            },
            {
                "role": "assistant",
                "content": "0"
            },
    
            # BORDERLINE TOXIC
            {
                "role": "user",
                "content": "Ns chuyện nghe đủ hiểu trình độ tới đâu rồi"
            },
            {
                "role": "assistant",
                "content": "1"
            },
    
            {
                "role": "user",
                "content": "Khôn như này quê tôi xích đầy"
            },
            {
                "role": "assistant",
                "content": "1"
            },
    
            # CLEAR TOXIC
            {
                "role": "user",
                "content": "Ngu thì im mẹ đi"
            },
            {
                "role": "assistant",
                "content": "1"
            },
    
            {
                "role": "user",
                "content": "Toàn một lũ phá hoại"
            },
            {
                "role": "assistant",
                "content": "1"
            },
    
            {
                "role": "user",
                "content": text
            }
        ]

In [10]:
class QwenModel:

    def __init__(self, model_name):

        self.model_name = model_name

        self.tokenizer = None
        self.model = None

    def load(self):

        self.tokenizer = AutoTokenizer.from_pretrained(
            self.model_name,
            trust_remote_code=True
        )

        self.model = AutoModelForCausalLM.from_pretrained(
            self.model_name,
            device_map="auto",
            dtype=(
                torch.float16
                if torch.cuda.is_available()
                else torch.float32
            ),
            trust_remote_code=True
        )

        self.model.eval()

        return self

In [11]:
class GenerativeClassifier:

    def __init__(
        self,
        model,
        tokenizer,
        labels
    ):

        self.model = model
        self.tokenizer = tokenizer

        # ví dụ:
        # binary  -> ["0", "1"]
        # 3-class -> ["0", "1", "2"]
        self.labels = [
            str(label)
            for label in labels
        ]

    @torch.no_grad()
    def score_label(
        self,
        prompt_messages,
        label
    ):

        prompt_text = self.tokenizer.apply_chat_template(
            prompt_messages,
            tokenize=False,
            add_generation_prompt=True
        )

        # thêm khoảng trắng để tokenization ổn định
        target_text = f"{label}"

        full_text = prompt_text + target_text

        full_enc = self.tokenizer(
            full_text,
            return_tensors="pt",
            add_special_tokens=False
        )

        prompt_enc = self.tokenizer(
            prompt_text,
            return_tensors="pt",
            add_special_tokens=False
        )

        input_ids = full_enc.input_ids.to(
            self.model.device
        )

        attention_mask = full_enc.attention_mask.to(
            self.model.device
        )

        labels_tensor = input_ids.clone()

        prompt_len = prompt_enc.input_ids.shape[1]

        # chỉ tính loss trên phần label
        labels_tensor[:, :prompt_len] = -100

        outputs = self.model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels_tensor
        )

        # negative loss = log-likelihood
        return -outputs.loss.item()

    @torch.no_grad()
    def predict(
        self,
        prompts
    ):

        all_preds = []
        all_probs = []

        for prompt in tqdm(prompts):

            scores = []

            for label in self.labels:

                score = self.score_label(
                    prompt_messages=prompt,
                    label=label
                )

                scores.append(score)

            scores = torch.tensor(
                scores,
                dtype=torch.float32
            )

            probs = F.softmax(
                scores,
                dim=-1
            )

            pred_idx = torch.argmax(
                probs
            ).item()

            # map về label thực
            pred_label = int(
                self.labels[pred_idx]
            )

            all_preds.append(pred_label)

            all_probs.append(
                probs.cpu().numpy()
            )

        return (
            np.array(all_preds),
            np.array(all_probs)
        )

In [12]:
class ClassificationEvaluator:

    @staticmethod
    def evaluate(
        y_true,
        y_pred,
        label_map,
        dataset_name
    ):

        print("\n")
        print("=" * 80)
        print(dataset_name)
        print("=" * 80)

        acc = accuracy_score(
            y_true,
            y_pred
        )

        mf1 = f1_score(
            y_true,
            y_pred,
            average="macro"
        )

        print(f"Accuracy : {acc:.4f}")
        print(f"MacroF1  : {mf1:.4f}")

        print("\nClassification Report:\n")

        print(
            classification_report(
                y_true,
                y_pred,
                target_names=list(
                    label_map.values()
                ),
                digits = 6,
            )
        )

In [13]:
vihsd = ViHSDProcessor().load()

victsd = ViCTSDProcessor().load()

README.md: 0.00B [00:00, ?B/s]

train.csv: 0.00B [00:00, ?B/s]

dev.csv: 0.00B [00:00, ?B/s]

test.csv: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/24048 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2672 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/6680 [00:00<?, ? examples/s]

ViCTSD.csv: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/10000 [00:00<?, ? examples/s]

In [14]:
model_wrapper = QwenModel(
    cfg.model_name
).load()

config.json:   0%|          | 0.00/728 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

In [15]:
vihsd_df = vihsd.get_test_dataframe()
vihsd_df = sample_dataframe(
    vihsd_df,
    ratio=cfg.sample_ratio,
    seed=cfg.seed
)

vihsd_prompts = [
    vihsd.build_prompt(x)
    for x in vihsd_df["text"]
]

classifier = GenerativeClassifier(
    model=model_wrapper.model,
    tokenizer=model_wrapper.tokenizer,
    labels=["0", "1", "2"]
)

vihsd_preds, vihsd_probs = classifier.predict(
    vihsd_prompts
)

ClassificationEvaluator.evaluate(
    vihsd_df["label"],
    vihsd_preds,
    vihsd.label_map,
    "ViHSD"
)

vihsd_results = pd.DataFrame({
    "text": vihsd_df["text"],
    "label": vihsd_df["label"],
    "prediction": vihsd_preds,
    "prob_clean": vihsd_probs[:, 0],
    "prob_offensive": vihsd_probs[:, 1],
    "prob_hate": vihsd_probs[:, 2]
})

vihsd_correct_samples = vihsd_results[
    vihsd_results["label"] == vihsd_results["prediction"]
].head(20)

vihsd_wrong_samples = vihsd_results[
    vihsd_results["label"] != vihsd_results["prediction"]
].head(20)

print("CORRECT")
display(vihsd_correct_samples)

print("INCORRECT")
display(vihsd_wrong_samples)

  0%|          | 0/6680 [00:00<?, ?it/s]



ViHSD
Accuracy : 0.7952
MacroF1  : 0.5516

Classification Report:

              precision    recall  f1-score   support

       CLEAN   0.932799  0.860671  0.895285      5548
   OFFENSIVE   0.254125  0.346847  0.293333       444
        HATE   0.401047  0.556686  0.466220       688

    accuracy                       0.795210      6680
   macro avg   0.529324  0.588068  0.551613      6680
weighted avg   0.832923  0.795210  0.811084      6680

CORRECT


,text,label,prediction,prob_clean,prob_offensive,prob_hate
0,Inoue Kido photoshop,0,0,0.736083,0.226253,0.037663
2,"Sáng đánh răng bị chảy máu chân răng, để lâu k...",0,0,0.857712,0.105691,0.036597
3,Phong cách thả rông😂,0,0,0.986365,0.013063,0.000571
4,From Cà Mau with love (P9),0,0,0.959730,0.036636,0.003635
5,-_- rủi thiết bị giám sát hay DTDD nó quay lại...,0,0,0.812526,0.152669,0.034805
6,Choán thực sự :(( chia lên chia xuống kb bao n...,0,0,0.896619,0.099815,0.003565
7,Khi bạn vừa ngủ dậy chưa kịp kéo quần thì phải...,0,0,0.738521,0.241643,0.019835
8,e chồng của Hà Tăng Chung Gia Phú Trang Huỳnh,0,0,0.973861,0.021348,0.004791
9,Tuần sau lại được nghỉ rồi 🤭,0,0,0.982742,0.015638,0.001620
10,Ghê vãi,1,1,0.115021,0.873456,0.011523


INCORRECT


,text,label,prediction,prob_clean,prob_offensive,prob_hate
1,Đóm ơi lên là lênnnn,0,1,0.436622,0.554102,0.009277
17,Thái Phan Chuẩn cmnr😂,0,1,0.327965,0.528197,0.143838
18,đm ông điêu ít thôiii :))),2,1,0.278643,0.706002,0.015356
20,Thâu tớm cả cáo 😂,0,1,0.394969,0.581450,0.023581
24,Có chửi chỉ dể xã tress chứ ai nghe đâu.,0,1,0.280192,0.410874,0.308935
28,Đm! Đen méo chơi nữa,0,1,0.168074,0.762135,0.069791
30,"chia ra thôi, bán miền bắc cho tq cho dễ =)))",0,2,0.026134,0.032588,0.941277
33,co lam. gi giau vay co day con lam voi con lam...,0,1,0.385033,0.524226,0.090742
41,Xã hội thối nát,2,0,0.354925,0.324427,0.320648
48,Người ta nói dân là sức mạnh của nước! Dần già...,0,2,0.034581,0.043459,0.921960


In [16]:
vihsd_correct_samples = (
    vihsd_results[
        vihsd_results["label"] == vihsd_results["prediction"]
    ]
    .sort_values(
        by="label",
        ascending=False
    )
    .head(20)
)
display(vihsd_correct_samples)

,text,label,prediction,prob_clean,prob_offensive,prob_hate
4320,Lua dao nhung nguoi nhe long .lua dao cac dan ...,2,2,0.120040,0.033595,0.846365
5698,Ở đéo đâu chả có ung thư :/ đời sống đi lên tỷ...,2,2,0.142616,0.065040,0.792345
5694,Dân chống nhưng quan chức không chống,2,2,0.260979,0.100616,0.638404
1102,"Cá chết vẫn chưa rõ nguyên nhân, biển thì vẫn ...",2,2,0.034164,0.065596,0.900239
1107,"Nhok Lầy cái nick thì clone, avatar thì đéo dá...",2,2,0.160390,0.377314,0.462295
1108,Con quỷ ngọc hiếu.. Cái thứ ngu ngục tù.. Đou ...,2,2,0.077664,0.093864,0.828472
4338,cả đoàn trốn trại...nói chuyện điên khùng,2,2,0.141778,0.288647,0.569575
1122,Quý vị đừng mong Việt nam phát triển nhé. Nhìn...,2,2,0.048102,0.011811,0.940087
3270,Theo trung cộng đi rồi đến lúc ko còn lợi dụng...,2,2,0.080027,0.025330,0.894644
2617,Bọn này luôn gây hiệu ứng đám đông để nôi tiến...,2,2,0.261663,0.175672,0.562664


In [17]:
victsd_df = victsd.get_test_dataframe()
victsd_df = sample_dataframe(
    victsd_df,
    ratio=cfg.sample_ratio,
    seed=cfg.seed
)

victsd_prompts = [
    victsd.build_prompt(x)
    for x in victsd_df["text"]
]

victsd_preds, victsd_probs = classifier.predict(
    victsd_prompts
)

ClassificationEvaluator.evaluate(
    victsd_df["label"],
    victsd_preds,
    victsd.label_map,
    "ViCTSD"
)

victsd_results = pd.DataFrame({
    "text": victsd_df["text"],
    "label": victsd_df["label"],
    "prediction": victsd_preds,
    "prob_non_toxic": victsd_probs[:, 0],
    "prob_toxic": victsd_probs[:, 1]
})

victsd_correct_samples = victsd_results[
    victsd_results["label"] == victsd_results["prediction"]
].head(20)

victsd_wrong_samples = victsd_results[
    victsd_results["label"] != victsd_results["prediction"]
].head(20)

print("CORRECT")
display(victsd_correct_samples)

print("INCORRECT")
display(victsd_wrong_samples)

Filter:   0%|          | 0/10000 [00:00<?, ? examples/s]

  0%|          | 0/1000 [00:00<?, ?it/s]



ViCTSD
Accuracy : 0.6740
MacroF1  : 0.5642

Classification Report:

              precision    recall  f1-score   support

   NON_TOXIC   0.960784  0.660674  0.782956       890
       TOXIC   0.221649  0.781818  0.345382       110

    accuracy                       0.674000      1000
   macro avg   0.591217  0.721246  0.564169      1000
weighted avg   0.879479  0.674000  0.734823      1000

CORRECT


,text,label,prediction,prob_non_toxic,prob_toxic
0,Nên có 1 bộ luật linh động gặp những trường hợ...,0,0,0.769451,0.230131
2,"Hổ dữ còn không ăn thịt con...Đồ khốn nạn, khô...",1,1,0.054977,0.944509
4,"Nissan đang giảm giá kìa bạn, bạn quất luôn ch...",0,0,0.825832,0.170753
5,"Quá nguy hiểm, việc làm thẻ tín dụng hay làm g...",0,0,0.815821,0.183462
6,Bài viết hay và nhiều thông tin! Cảm ơn tác gi...,0,0,0.945977,0.053160
7,"chả hề gì, 2 tuần sau ông ấy sẽ vượt qua thôi",0,0,0.602969,0.395437
9,"Phim đó tui xem khi đi thi đại học, tới giờ có...",0,0,0.898369,0.097503
11,Happy birthday! Happy Happy hạnh phúc,0,0,0.900261,0.098474
12,Chó và rắn đều không phải là thức ăn! Nhắn nhủ...,1,1,0.103706,0.895883
13,Các loài dưới biển còn có cơ hội lẫn sâu xuống...,0,0,0.678443,0.319849


INCORRECT


,text,label,prediction,prob_non_toxic,prob_toxic
1,Hởi các Cẩu Tặc hảy đọc bài nầy mà bỏ nghề....,0,1,0.060406,0.939384
3,Giọng ca thế này thì nghệ sĩ trẻ còn phải luyệ...,0,1,0.024648,0.974977
8,Dù con chó nào nhìn đẹp nhìn sạch đến đâu cũng...,0,1,0.249876,0.748913
10,"Trẻ em phải được vui chơi chạy nhảy, tại sao p...",1,0,0.731475,0.267000
14,Vậy mà có người đòi nhanh chóng có vaccine cov...,0,1,0.484637,0.512879
19,Tội nặng nhất là gieo vào đầu đứa bé tính trộm...,0,1,0.135425,0.862628
21,"Tuyệt, cách hay nhất để quên người cũ là tìm n...",0,1,0.390165,0.609039
22,Lây nhiễm là điều tất yếu. Đất nước thì dịch b...,0,1,0.279574,0.713917
23,"Ở Mỹ trốn thuế hơi bị khó, doanh nghiệp liên q...",0,1,0.319585,0.679209
27,Tôi sẽ không gây ra bất cứ sự xung đột nào với...,0,1,0.083202,0.915718
